# 06 · E2diag: widened-grid diagnostic (EXPLORATORY)

**Runtime:** T4 GPU. **Time:** about 20–30 minutes. **Resumable.** Logged in `deviations.md` (2026-11-12).

E2 (pre-registered) did not pass gate G2, and several selected values sat on the grid edge
(F: smallest β = π/4; C, E: largest λ = 100). This run repeats E2 for the linear maps with wider grids:
λ ∈ {1e-4 … 1e4} for every map, and 6 knob values per knob map (F β down to π/32, tanh s down to 1/16,
RFF σ up to 16√d). Records go to `runs/E2diag/`, separate from E2; nothing here replaces Table 2.

**Theory check (stated before running):** as β → 0, F becomes a random linear map of tanh(h), so F's
accuracy should rise toward the linear maps (B, and D at small s). If it does, the optical map works as
the physics predicts and the phase nonlinearity is what costs accuracy on this task. If it does not,
F loses information and must be debugged before E3.

In [ ]:
# ---- Setup (edit the two lines below once) ------------------------------------------
GITHUB_REPO = "surensagh/PhotonicsAI-Project"                    # repo that holds this code
WORKSPACE   = "/content/drive/MyDrive/optical_lm_workspace"      # splits, cache, runs, reports
# -------------------------------------------------------------------------------------
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["OPTLM_WORKSPACE"] = WORKSPACE
os.environ["HF_HOME"] = "/content/hf_cache"          # downloads stay on the fast local disk

# Private repo: add a GitHub token as a Colab secret named GH_TOKEN (key icon in the left bar).
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
REPO_DIR = "/content/" + GITHUB_REPO.split("/")[-1]
cmd = ["git", "clone", "-q", url, REPO_DIR] if not os.path.isdir(REPO_DIR) else ["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    msg = res.stderr.replace(token, "***") if token else res.stderr
    raise RuntimeError("git failed (check GITHUB_REPO / GH_TOKEN):\n" + msg)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("repo   :", REPO_DIR)
print("commit :", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip())
print("dirty  :", bool(subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"]).strip()))
print("workspace:", WORKSPACE)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!python scripts/run_e2.py --config configs/e2_diag.yaml

In [ ]:
print(open(os.path.join(WORKSPACE, "reports", "e2diag_table2.md"), encoding="utf-8").read())

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(WORKSPACE, "reports", "e2diag_fig2.png")))